# Data,data overview and preprocessing
> find called function in preprocessing

## Data 

**To run it** you need the corpus unpacked:

```bash
make data        # TED.tar -> data/raw/ + data/processed/
```

Every number printed here is reproducible on your machine, and `make verify` is what proves your
copy matches the team's.

In [ ]:
import sys
from pathlib import Path


ROOT = Path.cwd()
while not (ROOT / "preprocessing" / "align.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "preprocessing"))

import align

print("repository root:", ROOT)
print("raw corpus at  :", align.RAW)

## 1. The alignment contract

Six languages, three splits, and **every language has exactly the same number of raw lines in
every split**. That equality is not a coincidence of the data — it is the contract the whole
project rests on. The cell below verifies it and then shows how much of each language is actually
filled in.

In [ ]:
for split in ["train", "dev", "test"]:
    cols = {lang: align._raw(split, lang) for lang in align.PROJECT_LANGS}
    sizes = {lang: len(c) for lang, c in cols.items()}
    assert len(set(sizes.values())) == 1, f"alignment broken: {sizes}"
    print(f"{split}: {next(iter(sizes.values())):,} aligned positions, identical in every language")
    for lang, c in cols.items():
        filled = sum(1 for x in c if x.strip())
        print(f"    {lang}: {filled:>9,} filled  ({filled / len(c):6.1%})")
    print()

# `_raw` is align.py's own reader — the same one `positions()` uses. It is called directly here
# only because the line counts *are* the contract, and the public API deliberately hides lines.

### What those fill rates actually mean

**az is filled in 2.3% of the training positions; ru in 80.8%** — a 35x difference in available
text across the *same* 258,098 slots. This single fact is why the project exists, and the thing
to hold on to when reading every later number:

- az and be are the **low-resource** side by measurement, not by label: 5,946 usable sentences
  out of 258,098 positions.
- `tr` / `uk` / `ru` (70.7% / 42.0% / 80.8%) are 20–45x az's size — what makes them useful as
  helpers *and* dangerous as competitors.
- It is the mechanism `C5` measures: one fixed 8,000-slot vocabulary, filled by raw frequency,
  so whoever has more text takes more of it.

⚠️ dev and test are *proportionally* different from train (az 12.5% of test vs 2.3% of train). Do
not read the test set as a miniature of the training set.


## 2. A blank line is a placeholder, not dirty data

An empty line means **"this position has no translation in this language"**. Drop those lines and
every position index shifts — and **nothing will error**. That is why `align.py` exposes
*position* and never *line*.

In [ ]:
en = align._raw("train", "en")
az = align._raw("train", "az")

shown = 0
for i in range(5300, 5450):
    if not az[i].strip():
        print(f"position {i}")
        print(f"    en: {en[i][:80]}")
        print(f"    az: {az[i]!r}   <- empty: no Azerbaijani at this position")
        shown += 1
        if shown == 3:
            break
if not shown:
    print("no empty Azerbaijani position in this window — try a wider range")

The English at those positions is present and perfectly ordinary. The Azerbaijani is simply
absent — the talk was never translated. **Nothing about it looks like an error**, which is the
whole problem: if you read the file line by line and drop the empties, position `5300` becomes
position `5299` and every pair after it is off by one or more, silently.

## 3. The parallel corpora

Blank positions removed, this is what is actually available per language pair. These are the
counts the rest of the team should quote; they are also in `out/parallel/manifest.json`.

The nine pairs are grouped by purpose: `az-en` / `be-en` are the main line, `tr-en` / `uk-en` /
`ru-en` are training data for the helper languages, `be-uk` / `be-ru` are the direct comparison for
RQ1, and `az-tr` / `az-be` are auxiliary.

In [ ]:
splits = ["train", "dev", "test"]
print(f"{'pair':10s}" + "".join(f"{s:>12s}" for s in splits))
print("-" * 46)
for a, b in align.DELIVER_PAIRS:
    counts = [len(align.positions(s, [a, b])) for s in splits]
    print(f"{a}-{b:6s}" + "".join(f"{c:>12,}" for c in counts))

### Reading the pair counts

- **The two ends of this table differ by ~670x.** `ru-en` has 208,458 training pairs; `az-be` has
  310. Any statement of the form "we trained on TED" is hiding that.
- **The main line is small.** `az-en` 5,946 and `be-en` 4,509 training pairs. That is the entire
  signal for the languages the project cares about most.
- **`be` has two helper relationships (uk, ru); `az` has one (tr).** Their helper *volumes* are
  108,495 and 208,458 for be versus 182,470 for az. This asymmetry is not incidental — it is the
  natural experiment RQ1 is built on, so always compare az and be *per helper*, never in aggregate.
- **`az-be` (310 pairs) is not usable as training data.** It exists only as pivot material.

## 4. This is a **translation corpus**, and that changes how you read every result

The tar ships no translator metadata, so this has to be established from the text — and it can be.
TED transcripts carry stage directions like `(Applause)` and `(Laughter)`. A *native* corpus would
not contain them at all. Here they appear in the same positions across languages, **translated**:

```
en: (Applause) Thank you.
az: (Alqışlar) Təşəkkürlər.
```

The cell walks the whole training split, so give it a minute or two.

In [ ]:
import directionality

rows = directionality.inheritance_table("train", ["az", "be", "tr", "uk", "ru"])
print(f"{'lang':6s}{'positions':>12s}{'en has ()':>11s}{'same pos':>10s}{'inherited':>11s}")
print("-" * 50)
for r in rows:
    rate = r["stage_direction_inheritance"]
    print(f"{r['lang']:6s}{r['aligned_positions']:>12,}{r['en_with_stage_direction']:>11,}"
          f"{r['both_have_stage_direction']:>10,}{rate:>10.1%}")

### What that says, and why it is not a footnote

**Every non-English side of this corpus is a translation of the English side** — 94-99.6% of the
positions where English carries a stage direction carry one too, translated. A native Azerbaijani
corpus would not have `(Alqışlar)` in it at all. This is not a curiosity; it invalidates three
readings that would otherwise look natural:

**1. The two directions are not symmetric tasks.** `X→en` recovers the *source* of a translation,
and translationese is English-shaped — calques carry English word order across — and its reference
is the English original. `en→X` has to match one particular human translator's choices, because
that translation *is* what BLEU scores against. **So a BLEU gap between directions is not by
itself evidence of a data-volume gap**, which is how it is easy to read it.

**2. RQ1's "similarity" channel is confounded.** tr / uk / ru are translations of *the same
English sentences* as az and be. Part of their shared structure therefore comes from one shared
translation process, not from the languages being related. Any claim that helper languages help
"through similarity" has to rule this out first.

**3. C3's baseline is inflated.** Under `V2` — a vocabulary trained on az+be+en that has **never
seen Turkish** — az tokens still land on pieces tr also produces ~77% of the time. That is not
transfer; both sides simply carry the same Latin characters, punctuation, digits and loanwords.
`C6` and `directionality.py` are what separate that from real morphology, which is why the
defensible C3 reading is `C3_morph`, not `C3`.

## 5. How to read the data — and the one thing not to do

**Use `align.py`, or `src/data.py`'s `get_language_pairs` which delegates to it.** Both give you
`(position, src_text, tgt_text)`; the position is what lets a pair be joined to the pivot triples
and to the frozen `out/parallel/` artifacts.

> ⚠️ **`data/processed/` must only feed vocabulary training — never extract parallel corpora from
> it.** It drops blank lines, so every position index is already shifted, **and nothing will error.**
> `data/raw/` keeps them. This is the single easiest way to produce numbers that look fine and are
> wrong.

There are also **two tracks** and their BLEU is not comparable:

| Track | Tokenizer | Are our V1–V5 vocabularies relevant? |
|---|---|---|
| NLLB fine-tuning | NLLB's own SentencePiece | **No** — the vocabulary is fixed |
| From-scratch transformer | ours (V1–V5) | Yes |

In [ ]:
try:
    sys.path.insert(0, str(ROOT))
    from src.data import get_language_pairs

    pairs = get_language_pairs("test", "az", "en")
    print(f"{len(pairs):,} az-en test pairs, each carrying its position number")
    for pos, az_s, en_s in pairs[:3]:
        print(f"  [{pos:>5}] {az_s[:56]}")
        print(f"          {en_s[:56]}")
except ImportError as e:
    print("skipped — src/data.py imports `datasets`:", e)

## 6. Am I on the same data as everyone else?

You cannot answer that by eyeballing files — the corpus is not in the repository (CC-BY-NC-ND-4.0
forbids distributing derivatives), so everyone unpacks their own copy. This is what answers it:

```bash
make verify
```

It runs **two** checks, and both matter: the parallel corpora against `manifest.json` (are we on
the same *data*), and the vocabularies against `out/vocabs/index.json` (are we on the same
*segmentation*). Sharing one corpus does not stop two people from segmenting it differently — and
that failure is silent too.

```
OK    all 81 files match the manifest — you and the team share one dataset.
OK    all 14 vocabularies match the index — you and the team segment identically.
```

**Do not run downstream experiments until both are green.**

## 7. Do the measured numbers hold up? (which ones survive a setting change)

Everything above is a measurement, and every measurement rests on a setting somebody chose: how
many helper sentences define "shared", what counts as a productive affix, which split is
evaluated, how much corpus the allocation metric samples. If a conclusion moves when a setting
moves, it is a statement about the setting — and **nothing in the pipeline would ever say so**.

`preprocessing/sensitivity.py` re-runs the real `cost_analysis.py` under varied flags and reports
which quantities move. Reproduce with:

```bash
make sensitivity      # ~15-20 min -> out/sensitivity.md
```

**What it found, in order of importance:**

**1. One headline number was never well defined.** `C5` ("az's share of the vocabulary slots")
moves **4.79x** depending on how much of each corpus you sample:

| allocation sample | 5% | 10% | 25% | whole corpus |
|---|---|---|---|---|
| `C5` for `V1_az` | 0.127 | 0.224 | 0.394 | **0.610** |
| `C5` for `V3_az` | 0.0043 | 0.0043 | 0.0041 | 0.0043 |

The reason is in `cost_analysis.py`: the sample is a **prefix**, not a random draw, so it covers
different talks for each language and does not preserve the corpora's 57x size ratio. The
"68.8% → 1.5%" that has been quoted is one arbitrary point in that space; the whole-corpus reading
is **61% → 0.43%**. The default is now the whole corpus, and **a C5 number is meaningless without
its protocol**.

**2. C5 also measures the wrong thing.** It credits a slot to whichever language uses it *most
often* — absolute frequency, not ownership. The proof is the zero-point: under `V1_az`, a
vocabulary trained on Azerbaijani **alone**, az reaches only **61%**, not the ~1.0 the metric was
designed to give. So say "az's frequency dominates", never "az owns".

**3. The three conclusions we actually report are safe, and provably so.** `--min-suffix-support`
swept over 5 / 20 / 100 leaves these unchanged to four decimals:

| | 5 | 20 | 100 |
|---|---|---|---|
| `C3` (V3 az) | 0.7997 | 0.7997 | 0.7997 |
| `generic ÷ C3` | 53.6% | 53.6% | 53.6% |
| `morph = C3 − generic` | 0.3712 | 0.3712 | 0.3712 |

That is not luck: **these quantities do not use the threshold at all**. So "53.6% of V3's C3 is
free noise", "temperature sampling cuts it to 37.7%", and "V5's is the highest at 63.7%" cannot be
overturned by moving it. Two other settings (`--helper-lines`, `--split`) move them by at most
1.2 percentage points.

**4. The one quantity that does depend on a setting is `affix`, by construction.** The threshold
*defines* what "affix" means, so `affix ÷ generic` swings 2.21x across the sweep. Report `morph`
(threshold-free) for the claim, and quote the threshold whenever an affix figure appears.

> Scope: this sweeps the *analysis* settings only. Upstream choices — vocabulary size (8k alone),
> the language set, the `HELPER` mapping — are separate. The `HELPER` mapping is tested in
> `directionality.py`, which shows that adding uk/ru to az changes nothing (tr already covers it)
> and that cross-script sharing is ~0 (be↔tr morphology 0.002, against 0.457 for az↔tr).

## Where everything lives

| | |
|---|---|
| `preprocessing/align.py` | the single data entry point — `positions` / `pairs` / `triples` |
| `preprocessing/directionality.py` | the translation-corpus evidence, and the corrected C3 |
| `preprocessing/sensitivity.py` | which measured numbers survive a setting change |
| `out/parallel/manifest.json` | per-file line counts + sha256 — **committed** |
| `out/vocabs/index.json` | per-vocabulary hashes + construction args — **committed** |
| `data/`, `out/` (everything else) | never committed; regenerate with `make all` |
| `docs/DELIVERY.md` | how to consume these artifacts |
| `README.md` | the vocabulary experiments (V1–V5, C1–C6) |